# LangChain Runnable

**Goal:** Understand the `Runnable` interface, why it exists, and how to build powerful LLM pipelines using `RunnableLambda`, `RunnableSequence`, `RunnableParallel`, and more.

**Prerequisite:** A `.env` file with `GROQ_API_KEY` set.

In [ ]:
# Setup: Load environment variables and install dependencies
# !pip install langchain langchain-groq python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()

# Verify the key is loaded
api_key = os.getenv("GROQ_API_KEY")
print("API Key loaded:", "Yes" if api_key else "No")

---

## 1. The Problem: No Common Interface

Before `Runnable`, every LangChain component had its own method name:

| Component | Method |
|-----------|--------|
| LLM | `.predict()` |
| PromptTemplate | `.format()` |
| Retriever | `.get_relevant_documents()` |
| Parser | `.parse()` |

**The Issue:** 4 different method names = no common way to connect them.

You had to write glue code every time you wanted to wire components together. It was like having 4 different chargers — none fit each other.

```
┌─────────────┐     ┌─────────────────┐     ┌─────────────┐     ┌─────────────┐
│    LLM      │     │  PromptTemplate │     │  Retriever  │     │   Parser    │
│             │     │                 │     │             │     │             │
│ .predict()  │     │   .format()     │     │.get_relevant│     │   .parse()  │
│             │     │                 │     │ _documents()│     │             │
└─────────────┘     └─────────────────┘     └─────────────┘     └─────────────┘

        4 different method names = no common way to connect them
```

---

## 2. The Fix: The Runnable Interface

LangChain introduced a **standard contract** called `Runnable`.

Every component — Prompt, Model, Parser, Retriever, or even your custom Python function — implements the same method:

```
.invoke(input) → output
```

**Analogy:** Think of Lego blocks. Every block has the same connector size, regardless of color. Red, blue, green — all fit together. `Runnable` made every LangChain component have the same connector: `.invoke()`.

```
        ┌─────────────────────────────────────────┐
        │           RUNNABLE CONTRACT             │
        │                                         │
        │         .invoke(input) → output         │
        │                                         │
        └─────────────────────────────────────────┘
                      ↑       ↑       ↑
        ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
        │   Runnable  │   │   Runnable  │   │   Runnable  │
        │  .invoke()  │   │  .invoke()  │   │  .invoke()  │
        └─────────────┘   └─────────────┘   └─────────────┘
           Prompt            Model            Parser

   Same method name everywhere → they can now connect
```

---

## 3. The Pipe: LangChain Expression Language (LCEL)

Because every component shares `.invoke()`, they can be chained using the **pipe operator** `|`.

```python
chain = prompt | model | parser
```

This creates a `RunnableSequence` — where the output of step 1 becomes the input of step 2, and so on.

**ASCII Diagram:**
```
   INPUT ──→ [  Prompt  ] ──→ [  Model  ] ──→ [ Parser ] ──→ OUTPUT
                │                │               │
             .invoke()        .invoke()       .invoke()

   Code:  chain = prompt | model | parser
   Run:   chain.invoke({"topic": "AI"})
```

---

## 4. Step-by-Step: Building a Chain

### Step 1: Prove `.invoke()` works on a Prompt

A `PromptTemplate` is a `Runnable`. When you call `.invoke()`, it formats the template.

In [ ]:
from langchain_core.prompts import PromptTemplate

# PromptTemplate is a Runnable — it has .invoke()
prompt = PromptTemplate.from_template("Write a joke about {topic}")

# .invoke() formats the template with the given input
result = prompt.invoke({"topic": "AI"})
print(result)

### Step 2: Full Pipe Chain (RunnableSequence)

Now wire Prompt → Model → Parser using `|`.

**Real-world use case:** Customer support automation — raw message → summarize → classify category → draft reply. Each step depends on the previous one.

**Another example:** Legal contract analysis — analyze clause → extract risks → generate financial memo. The memo cannot exist without the risk analysis.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

# Step 1: Prompt (Runnable)
prompt = PromptTemplate.from_template("Write a short joke about {topic}")

# Step 2: Parser (Runnable)
parser = StrOutputParser()

# Pipe them together → RunnableSequence
chain = prompt | llm | parser

# One .invoke() runs the entire sequence
result = chain.invoke({"topic": "artificial intelligence"})
print(result)

---

## 5. RunnableLambda: Your Custom Functions in the Chain

What if you need custom logic that LangChain does not provide?

Examples:
- Count words in generated text
- Clean or validate output
- Transform data between steps

`RunnableLambda` wraps any normal Python function so it becomes a `Runnable` — supporting `.invoke()` and fitting into any chain with `|`.

In [ ]:
from langchain_core.runnables import RunnableLambda

# A normal Python function
def to_uppercase(text: str) -> str:
    return text.upper()

# Wrap it as a Runnable
uppercase_runnable = RunnableLambda(to_uppercase)

# Now it supports .invoke()
result = uppercase_runnable.invoke("hello world")
print(result)

### Using RunnableLambda Inside a Chain

**Real-world use case:** E-commerce product descriptions — generate description with LLM → count words → if too long, trim. The word-count step is custom logic embedded in the pipeline.

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())

# Custom function as Runnable
counter = RunnableLambda(word_count)

# Chain: prompt → model → parser → word counter
chain_with_counter = prompt | llm | parser | counter

result = chain_with_counter.invoke({"topic": "robots"})
print(f"The joke contains {result} words.")

---

## 6. RunnableParallel: Run Independent Tasks Concurrently

Sometimes you have multiple tasks that use the **same input** but are **independent** of each other.

Instead of running them one by one (slow), run them in parallel (fast).

`RunnableParallel` takes a dictionary of Runnables, runs them concurrently, and returns a dictionary of results.

**ASCII Diagram:**
```
                    ┌──→ [ Joke Chain ] ──┐
   INPUT ───────────┤                      ├──→ {"joke": ..., "fact": ...}
                    └──→ [ Fact Chain ] ──┘

   Same input → multiple independent Runnables → dictionary output
```

### Real-World Use Cases for RunnableParallel

| Scenario | Task A | Task B |
|----------|--------|--------|
| **Product Review Analysis** | Extract sentiment | Generate summary |
| **Resume Screening** | Extract skills | Summarize experience |
| **News Article** | Generate headline | Translate to Hindi |
| **Customer Feedback** | Detect urgency | Categorize topic |

All tasks receive the same input and run at the same time.

In [ ]:
from langchain_core.runnables import RunnableParallel

# Reuse the same LLM and parser
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
parser = StrOutputParser()

# Chain A: Generate a joke
joke_chain = (
    PromptTemplate.from_template("Write a short joke about {topic}")
    | llm
    | parser
)

# Chain B: Generate a fact
fact_chain = (
    PromptTemplate.from_template("Give one interesting fact about {topic}")
    | llm
    | parser
)

# Run both in parallel
parallel_chain = RunnableParallel(
    joke=joke_chain,
    fact=fact_chain
)

result = parallel_chain.invoke({"topic": "artificial intelligence"})
print(result)

---

## 7. RunnablePassthrough: Pass Input Through Unchanged

Sometimes you want to pass the original input forward while also processing it.

`RunnablePassthrough()` simply returns the input as-is. It is useful when you need to preserve the original data for later steps.

**Real-world use case:** You want to classify an email AND keep the original email text for the final report.

**Another use case:** Pass the user's original question to a retriever while also sending it to the LLM for rephrasing.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# This simply returns the input unchanged
passthrough = RunnablePassthrough()

result = passthrough.invoke("This text goes through unchanged")
print(result)

# Common pattern: keep original input while processing
chain = (
    {"original": RunnablePassthrough(), "processed": RunnableLambda(lambda x: x.upper())}
    | RunnableParallel(
        original=lambda x: x["original"],
        processed=lambda x: x["processed"]
    )
)

result = chain.invoke("hello")
print(result)

---

## 8. RunnableAssign: Add New Fields to a Dictionary

`RunnableAssign` takes a dictionary of Runnables and adds their outputs as new keys to the existing dictionary.

**Real-world use case:** Start with a customer ID → fetch profile → add `profile` key → fetch order history → add `orders` key. Each step enriches the data dictionary.

In [ ]:
from langchain_core.runnables import RunnableAssign

# Start with some base data
base_data = {"topic": "space travel"}

# Add a 'joke' field by running a chain
joke_prompt = PromptTemplate.from_template("Write a joke about {topic}")
joke_chain = joke_prompt | llm | parser

# RunnableAssign adds the result as a new key
enricher = RunnableAssign({"joke": joke_chain})

result = enricher.invoke(base_data)
print(result)

---

## 9. RunnableBranch: Conditional Routing

`RunnableBranch` lets you define **if-else logic** inside a chain.

It takes a list of `(condition, runnable)` pairs. The first condition that returns `True` determines which runnable runs.

**Real-world use case:** Customer support routing —
- If message contains "refund" → route to billing team chain
- If message contains "bug" → route to technical team chain
- Else → route to general inquiry chain

In [ ]:
from langchain_core.runnables import RunnableBranch

# Define condition functions and their corresponding chains
billing_chain = PromptTemplate.from_template(
    "You are a billing assistant. Help with: {query}"
) | llm | parser

tech_chain = PromptTemplate.from_template(
    "You are a technical support agent. Help with: {query}"
) | llm | parser

general_chain = PromptTemplate.from_template(
    "You are a general assistant. Help with: {query}"
) | llm | parser

# Branch: condition → runnable
router = RunnableBranch(
    (lambda x: "refund" in x["query"].lower(), billing_chain),
    (lambda x: "bug" in x["query"].lower() or "error" in x["query"].lower(), tech_chain),
    general_chain  # default fallback
)

# Test with different queries
for query in ["I want a refund", "There is a bug in login", "What are your hours?"]:
    result = router.invoke({"query": query})
    print(f"Query: {query}")
    print(f"Response: {result}")
    print("-" * 40)

---

## 10. RunnableMap: Transform Dictionary Keys

`RunnableMap` (or simply a Python `dict` in a chain) maps each key through its own Runnable.

It is the building block of `RunnableParallel`. When you write:

```python
{"a": chain1, "b": chain2}
```

LangChain automatically treats this as a `RunnableMap` / `RunnableParallel`.

**Real-world use case:** A search pipeline where you need to query a vector database, a keyword search engine, and a web API — all from the same user query — then merge the results.

In [ ]:
# A dict in a chain is treated as RunnableMap / RunnableParallel
from langchain_core.runnables import RunnableMap

# Define two parallel chains
sentiment_chain = (
    PromptTemplate.from_template("Classify sentiment as Positive or Negative: {review}")
    | llm | parser
)

summary_chain = (
    PromptTemplate.from_template("Summarize this review in one line: {review}")
    | llm | parser
)

# RunnableMap processes each key independently
review_analyzer = RunnableMap({
    "sentiment": sentiment_chain,
    "summary": summary_chain
})

review = "The product quality is really good, delivery was fast, but packaging could improve."
result = review_analyzer.invoke({"review": review})
print(result)

---

## 11. RunnableSequence: Chaining with Dependencies

`RunnableSequence` is what you get when you use `|`. It guarantees that step N only starts after step N-1 finishes.

**Key property:** Output of step 1 → Input of step 2 → Input of step 3...

**Real-world use case — Document Q&A Pipeline:**
1. Receive user question
2. Rephrase question for better retrieval
3. Retrieve relevant documents
4. Format documents + question into final prompt
5. Send to LLM
6. Parse and return answer

Each step depends on the previous. No shortcuts.

In [ ]:
# Step 1: Rephrase the question
rephrase_prompt = PromptTemplate.from_template(
    "Rephrase this question to be more specific and clear: {question}"
)
rephraser = rephrase_prompt | llm | parser

# Step 2: Answer the rephrased question
answer_prompt = PromptTemplate.from_template(
    "Answer this question concisely: {question}"
)
answerer = answer_prompt | llm | parser

# Full sequence: rephrase → answer
# We need to map the output of rephraser to the input of answerer
full_chain = (
    {"question": rephraser}  # rephraser output becomes 'question' for answerer
    | answerer
)

result = full_chain.invoke({"question": "Tell me about AI"})
print(result)

---

## 12. Complete Reference: All LangChain Runnables

| Runnable | What It Does | Real-World Use Case |
|----------|--------------|---------------------|
| **`Runnable`** | Base interface / contract. Every component implements `.invoke(input) → output`. | Building custom components that plug into any LangChain pipeline. |
| **`RunnableLambda`** | Wraps a normal Python function into a Runnable. | Word counting, text cleaning, custom validation, data transformation between chain steps. |
| **`RunnableSequence`** | Chains Runnables sequentially using `\|`. Output of step N → input of step N+1. | Customer support: summarize → classify → draft reply. Legal: analyze clause → extract risks → generate memo. |
| **`RunnableParallel`** | Runs multiple Runnables concurrently on the same input. Returns a dictionary. | Product review analysis: extract sentiment AND summary at the same time. Resume screening: extract skills AND experience summary together. |
| **`RunnablePassthrough`** | Returns the input unchanged. Useful for preserving original data. | Keep the original user query while also sending it to a retriever or rephraser. Pass email text forward while classifying it. |
| **`RunnableAssign`** | Adds new keys to an existing dictionary by running Runnables. | Enrich customer data: start with ID → add profile → add order history → add recommendations. |
| **`RunnableBranch`** | Conditional routing — first matching condition runs its Runnable. | Customer support routing: "refund" → billing chain, "bug" → tech chain, else → general chain. Content moderation: route toxic vs safe messages to different handlers. |
| **`RunnableMap`** | Maps each dictionary key through its own Runnable (dict literal in chains). | Multi-source search: query vector DB, keyword search, and web API in parallel from the same input. |
| **`RunnableBinding`** | Binds default arguments (like `temperature`, `callbacks`) to a Runnable. | Create a frozen copy of a chain with specific config (e.g., always use temperature=0 for factual tasks). |
| **`RunnableGenerator`** | Wraps a Python generator function as a Runnable. | Streaming token-by-token output from an LLM through a custom processing function. |
| **`RunnableRetry`** | Adds retry logic with backoff to any Runnable. | API call fails? Auto-retry 3 times before giving up. Useful for unreliable third-party services. |
| **`RunnableFallbacks`** | Defines fallback Runnables if the primary one fails. | Primary LLM (GPT-4) is down? Fall back to Groq. Groq fails? Fall back to local model. |
| **`RunnableConfigurable`** | Makes a Runnable configurable at runtime via `configurable_fields`. | Same chain, but swap the LLM model or prompt template at runtime without rewriting code. |
| **`RunnablePick`** | Picks specific keys from a dictionary output. | After a parallel chain returns 5 keys, extract only the 2 keys you need for the next step. |
| **`RunnableEach`** | Applies a Runnable to each item in a list input. | Batch process: send 10 customer reviews through the same sentiment analyzer, one by one. |
| **`RunnableWithMessageHistory`** | Adds chat message history management to a Runnable. | Build a chatbot that remembers previous turns without manually managing history in every invoke call. |
| **`RunnableConfig`** | Configuration object passed to `.invoke()` for callbacks, metadata, recursion limits. | Add tracing tags, set max recursion depth, or attach custom callbacks for logging. |


---

## Key Takeaways

1. **`.invoke(input) → output`** is the universal contract. Every component speaks the same language.
2. **`|` (pipe)** builds `RunnableSequence` — sequential, dependent steps.
3. **`RunnableParallel`** (or `dict` literal) runs independent tasks concurrently.
4. **`RunnableLambda`** lets you inject any custom Python logic into the chain.
5. **`RunnableBranch`** adds if-else routing without leaving the chain.
6. **`RunnablePassthrough`** preserves original data while processing it.
7. **All Runnables are composable** — you can nest them infinitely: parallel inside sequence, sequence inside branch, lambda inside parallel, etc.

> **Remember:** If you can write it as a function, you can make it a `Runnable`. If you can make it a `Runnable`, you can pipe it, parallelize it, branch it, or retry it.

---

## Exercise: Build a Multi-Task Pipeline

Build a chain that does the following:

1. Takes a `{topic}` as input.
2. **In parallel**, generates:
   - A joke about the topic
   - A one-line summary about the topic
   - The word count of the topic name (use `RunnableLambda`)
3. **Then sequentially**, takes the joke and explains why it is funny.

**Hint:** Use `RunnableParallel` for step 2, then pipe the result into a `RunnableSequence` that formats the joke for explanation.

In [ ]:
# Exercise Solution
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)
parser = StrOutputParser()

# Step 2: Parallel generation
parallel = RunnableParallel(
    joke=(
        PromptTemplate.from_template("Write a short joke about {topic}")
        | llm | parser
    ),
    summary=(
        PromptTemplate.from_template("Describe {topic} in one sentence.")
        | llm | parser
    ),
    word_count=RunnableLambda(lambda x: len(x["topic"].split()))
)

# Step 3: Explain the joke (depends on parallel output)
explain_prompt = PromptTemplate.from_template(
    "Explain why this joke is funny: {joke}"
)
explainer = explain_prompt | llm | parser

# Full pipeline: parallel → extract joke → explain
full_pipeline = parallel | {"joke": lambda x: x["joke"]} | explainer

result = full_pipeline.invoke({"topic": "programming"})
print(result)